In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
import yaml

In [2]:
with open('../config.yaml', 'r') as file:
    config = yaml.safe_load(file)

In [3]:
final_demo = pl.read_csv(config["input_data"]["file1"])
final_demo.head()

client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth
i64,f64,f64,f64,str,f64,f64,f64,f64
836976,6.0,73.0,60.5,"""U""",2.0,45105.3,6.0,9.0
2304905,7.0,94.0,58.0,"""U""",2.0,110860.3,6.0,9.0
1439522,5.0,64.0,32.0,"""U""",2.0,52467.79,6.0,9.0
1562045,16.0,198.0,49.0,"""M""",2.0,67454.65,3.0,6.0
5126305,12.0,145.0,33.0,"""F""",2.0,103671.75,0.0,3.0


In [4]:
experiment_data = pl.read_csv(config["input_data"]["file2"])
experiment_data.head()

client_id,Variation
i64,str
9988021,"""Test"""
8320017,"""Test"""
4033851,"""Control"""
1982004,"""Test"""
9294070,"""Control"""


In [5]:
webdata1 = pl.read_csv(config["input_data"]["file3"])
webdata1.head()

client_id,visitor_id,visit_id,process_step,date_time
i64,str,str,str,str
9988021,"""580560515_7732621733""","""781255054_21935453173_531117""","""step_3""","""2017-04-17 15:27:07"""
9988021,"""580560515_7732621733""","""781255054_21935453173_531117""","""step_2""","""2017-04-17 15:26:51"""
9988021,"""580560515_7732621733""","""781255054_21935453173_531117""","""step_3""","""2017-04-17 15:19:22"""
9988021,"""580560515_7732621733""","""781255054_21935453173_531117""","""step_2""","""2017-04-17 15:19:13"""
9988021,"""580560515_7732621733""","""781255054_21935453173_531117""","""step_3""","""2017-04-17 15:18:04"""


In [6]:
webdata2 = pl.read_csv(config["input_data"]["file2"])
webdata2.head()

client_id,Variation
i64,str
9988021,"""Test"""
8320017,"""Test"""
4033851,"""Control"""
1982004,"""Test"""
9294070,"""Control"""


In [7]:
webdata2["Variation"].value_counts()

Variation,count
str,u32
"""Test""",26968
"""Control""",23532
"""NA""",20109


In [8]:
webdata1["process_step"].value_counts().sort("count", descending=True)

process_step,count
str,u32
"""start""",108910
"""step_1""",73432
"""step_2""",61768
"""step_3""",53628
"""confirm""",45403


MERGED DATA

In [9]:
merged = pl.read_csv(config["output_data"]["file5"])
pl.DataFrame(merged).head()

,client_id,visitor_id,visit_id,process_step,date_time,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,Variation
i64,i64,str,str,str,str,f64,f64,f64,str,f64,f64,f64,f64,str
0,9988021,"""580560515_7732621733""","""781255054_21935453173_531117""","""step_3""","""2017-04-17 15:27:07""",5.0,64.0,79.0,"""U""",2.0,189023.86,1.0,4.0,"""Test"""
1,9988021,"""580560515_7732621733""","""781255054_21935453173_531117""","""step_2""","""2017-04-17 15:26:51""",5.0,64.0,79.0,"""U""",2.0,189023.86,1.0,4.0,"""Test"""
2,9988021,"""580560515_7732621733""","""781255054_21935453173_531117""","""step_3""","""2017-04-17 15:19:22""",5.0,64.0,79.0,"""U""",2.0,189023.86,1.0,4.0,"""Test"""
3,9988021,"""580560515_7732621733""","""781255054_21935453173_531117""","""step_2""","""2017-04-17 15:19:13""",5.0,64.0,79.0,"""U""",2.0,189023.86,1.0,4.0,"""Test"""
4,9988021,"""580560515_7732621733""","""781255054_21935453173_531117""","""step_3""","""2017-04-17 15:18:04""",5.0,64.0,79.0,"""U""",2.0,189023.86,1.0,4.0,"""Test"""


In [12]:
import polars as pl
import yaml

df = pl.read_csv(config["output_data"]["file5"])

STEPS = ['start', 'step_1', 'step_2', 'step_3', 'confirm']
STEP_ORDER = {step: i for step, i in zip(STEPS, range(len(STEPS)))}
# {'start': 0, 'step_1': 1, 'step_2': 2, 'step_3': 3, 'confirm': 4}

# Sort chronologically within each visit
df_sorted = df.sort(['visit_id', 'date_time'])

# For each row, get the NEXT step and NEXT date_time within the same visit_id
df_transitions = (
    df_sorted
    .with_columns([
        pl.col('process_step').shift(-1).over('visit_id').alias('next_step'),
        pl.col('date_time').shift(-1).over('visit_id').alias('next_date_time'),
    ])
    # Drop the last row of each visit (no "next" step exists)
    .filter(pl.col('next_step').is_not_null())
)

# Map each step to its order index
df_transitions = df_transitions.with_columns([
    pl.col('process_step').replace(STEP_ORDER).cast(pl.Int8).alias('step_rank'),
    pl.col('next_step').replace(STEP_ORDER).cast(pl.Int8).alias('next_step_rank'),
])

# Flag errors
df_transitions = df_transitions.with_columns([
    (pl.col('process_step') == pl.col('next_step')).alias('is_repeat'),
    (pl.col('next_step_rank') < pl.col('step_rank')).alias('is_regression'),
]).with_columns(
    (pl.col('is_repeat') | pl.col('is_regression')).alias('has_error')
)

# Label the transition for readability
df_transitions = df_transitions.with_columns(
    (pl.col('process_step') + ' -> ' + pl.col('next_step')).alias('transition')
)

# Final selection — keep date_times for future time-spent analysis
df_transitions = df_transitions.select([
    'visit_id',
    'Variation',
    'process_step',
    'date_time',
    'next_step',
    'next_date_time',
    'transition',
    'is_repeat',
    'is_regression',
    'has_error'
])

# --- Summary: which transitions are most problematic? ---
error_summary = (
    df_transitions
    .filter(pl.col('has_error'))
    .group_by('transition')
    .agg([
        pl.len().alias('error_count'),
        pl.col('is_repeat').sum().alias('repeats'),
        pl.col('is_regression').sum().alias('regressions'),
    ])
    .sort('error_count', descending=True)
)

print("=== Transition Error Summary ===")
print(error_summary)

print(f"\n=== Total transitions: {len(df_transitions)} ===")
print(f"=== Transitions with errors: {df_transitions['has_error'].sum()} ===")
print(f"=== Visit_ids with at least one error: {df_transitions.filter(pl.col('has_error'))['visit_id'].n_unique()} ===")

=== Transition Error Summary ===
shape: (15, 4)
┌────────────────────┬─────────────┬─────────┬─────────────┐
│ transition         ┆ error_count ┆ repeats ┆ regressions │
│ ---                ┆ ---         ┆ ---     ┆ ---         │
│ str                ┆ u32         ┆ u32     ┆ u32         │
╞════════════════════╪═════════════╪═════════╪═════════════╡
│ start -> start     ┆ 15300       ┆ 15300   ┆ 0           │
│ step_1 -> start    ┆ 6054        ┆ 0       ┆ 6054        │
│ step_3 -> step_2   ┆ 3792        ┆ 0       ┆ 3792        │
│ confirm -> confirm ┆ 3621        ┆ 3621    ┆ 0           │
│ step_2 -> step_1   ┆ 3034        ┆ 0       ┆ 3034        │
│ …                  ┆ …           ┆ …       ┆ …           │
│ step_3 -> step_1   ┆ 978         ┆ 0       ┆ 978         │
│ confirm -> start   ┆ 838         ┆ 0       ┆ 838         │
│ confirm -> step_1  ┆ 126         ┆ 0       ┆ 126         │
│ confirm -> step_3  ┆ 105         ┆ 0       ┆ 105         │
│ confirm -> step_2  ┆ 1           ┆ 

In [16]:
df = pl.read_csv(config["output_data"]["file5"])

STEPS = ['start', 'step_1', 'step_2', 'step_3', 'confirm']
STEP_ORDER = {step: i for step, i in zip(STEPS, range(len(STEPS)))}

df = df.with_columns(pl.col('date_time').str.to_datetime(format='%Y-%m-%d %H:%M:%S'))
df_sorted = df.sort(['visit_id', 'date_time'])

# --- For each visit_id, keep only the FIRST occurrence of each step (regression rule) ---
# Then for repeats, we sum time — but since we're keeping first occurrence as start,
# the "end" of a step is the first occurrence of the NEXT step in STEPS order.

# Step 1: tag each row with the step rank
df_ranked = df_sorted.with_columns(
    pl.col('process_step')
    .replace(STEP_ORDER)
    .cast(pl.Int8)
    .alias('step_rank')
)

# Step 2: keep only the first occurrence of each (visit_id, process_step) pair
# This handles both the regression rule (first entry wins) and cleans repeats
df_first = (
    df_ranked
    .group_by(['visit_id', 'process_step'])
    .agg([
        pl.col('date_time').min().alias('step_start'),  # first entry = start time
        pl.col('step_rank').first().alias('step_rank'),
        pl.col('Variation').first().alias('Variation'),
    ])
    .sort(['visit_id', 'step_rank'])
)

# Step 3: get the start time of the NEXT step within the same visit_id
df_first = df_first.with_columns(
    pl.col('step_start').shift(-1).over('visit_id').alias('next_step_start')
)

# Step 4: drop only rows where the CURRENT step is the last (no next step exists)
# i.e. keep step_3 -> confirm by checking next_step_start is not null AND
# the current step is not 'confirm' (confirm has no outgoing transition anyway)
df_durations = (
    df_first
    .filter(
        pl.col('next_step_start').is_not_null() & 
        (pl.col('process_step') != 'confirm')
    )
    .with_columns(
        (pl.col('next_step_start') - pl.col('step_start'))
        .dt.total_seconds()
        .alias('duration_seconds')
    )
    .filter(pl.col('duration_seconds') >= 0)
)

# Step 5: label the transition
df_durations = df_durations.with_columns(
    pl.col('process_step').shift(-1).over('visit_id').alias('next_step')
).with_columns(
    (pl.col('process_step') + ' -> ' + pl.col('next_step')).alias('transition')
)

# --- Average time per transition ---
avg_time = (
    df_durations
    .group_by('transition')
    .agg([
        pl.col('duration_seconds').mean().alias('avg_seconds'),
        pl.col('duration_seconds').median().alias('median_seconds'),
        pl.col('duration_seconds').std().alias('std_seconds'),
        pl.len().alias('n_visits')
    ])
    .with_columns([
        (pl.col('avg_seconds') / 60).round(2).alias('avg_minutes'),
        (pl.col('median_seconds') / 60).round(2).alias('median_minutes'),
    ])
    .sort('transition')
)

print("=== Average Time per Transition ===")
print(avg_time.select([
    'transition', 'n_visits',
    'avg_minutes', 'median_minutes', 'std_seconds'
]))

=== Average Time per Transition ===
shape: (7, 5)
┌──────────────────┬──────────┬─────────────┬────────────────┬─────────────┐
│ transition       ┆ n_visits ┆ avg_minutes ┆ median_minutes ┆ std_seconds │
│ ---              ┆ ---      ┆ ---         ┆ ---            ┆ ---         │
│ str              ┆ u32      ┆ f64         ┆ f64            ┆ f64         │
╞══════════════════╪══════════╪═════════════╪════════════════╪═════════════╡
│ null             ┆ 40084    ┆ 2.23        ┆ 1.07           ┆ 243.248075  │
│ start -> step_1  ┆ 33559    ┆ 0.99        ┆ 0.28           ┆ 288.881384  │
│ start -> step_2  ┆ 30       ┆ 1.11        ┆ 0.39           ┆ 98.087083   │
│ start -> step_3  ┆ 44       ┆ 13.03       ┆ 11.31          ┆ 642.606972  │
│ step_1 -> step_2 ┆ 30582    ┆ 1.07        ┆ 0.48           ┆ 155.522073  │
│ step_1 -> step_3 ┆ 49       ┆ 5.6         ┆ 2.9            ┆ 463.741396  │
│ step_2 -> step_3 ┆ 25565    ┆ 1.81        ┆ 1.25           ┆ 181.549247  │
└──────────────────┴──────

In [ ]:
TESTING

In [21]:
df = pl.read_csv(config["output_data"]["file5"])

STEPS = ['start', 'step_1', 'step_2', 'step_3', 'confirm']
STEP_ORDER = {step: i for step, i in zip(STEPS, range(len(STEPS)))}

df = df.with_columns(pl.col('date_time').str.to_datetime(format='%Y-%m-%d %H:%M:%S'))
df_sorted = df.sort(['visit_id', 'date_time'])

In [22]:
# For every row, look at what came before it (within the same visit_id)
df_transitions = df_sorted.with_columns([
    pl.col('process_step').shift(1).over('visit_id').alias('prev_step'),
    pl.col('date_time').shift(1).over('visit_id').alias('prev_date_time'),
    pl.col('process_step').replace(STEP_ORDER).cast(pl.Int8).alias('step_rank'),
]).with_columns(
    pl.col('prev_step').replace(STEP_ORDER).cast(pl.Int8).alias('prev_step_rank')
)

# Drop the first row of each visit (no previous step exists)
df_transitions = df_transitions.filter(pl.col('prev_step').is_not_null())

# Label the transition
df_transitions = df_transitions.with_columns(
    (pl.col('prev_step') + ' -> ' + pl.col('process_step')).alias('transition')
)

In [23]:
df_transitions = df_transitions.with_columns([
    (pl.col('process_step') == pl.col('prev_step')).alias('is_repeat'),
    (pl.col('step_rank') < pl.col('prev_step_rank')).alias('is_regression'),
]).with_columns(
    (pl.col('is_repeat') | pl.col('is_regression')).alias('has_error')
)

error_summary = (
    df_transitions
    .filter(pl.col('has_error'))
    .group_by('transition')
    .agg([
        pl.len().alias('error_count'),
        pl.col('is_repeat').sum().alias('repeats'),
        pl.col('is_regression').sum().alias('regressions'),
    ])
    .sort('error_count', descending=True)
)

print("=== Transition Error Summary ===")
print(error_summary)
print(f"\nTotal transitions: {len(df_transitions)}")
print(f"Transitions with errors: {df_transitions['has_error'].sum()}")
print(f"Visit_ids with at least one error: {df_transitions.filter(pl.col('has_error'))['visit_id'].n_unique()}")

=== Transition Error Summary ===
shape: (15, 4)
┌────────────────────┬─────────────┬─────────┬─────────────┐
│ transition         ┆ error_count ┆ repeats ┆ regressions │
│ ---                ┆ ---         ┆ ---     ┆ ---         │
│ str                ┆ u32         ┆ u32     ┆ u32         │
╞════════════════════╪═════════════╪═════════╪═════════════╡
│ start -> start     ┆ 15300       ┆ 15300   ┆ 0           │
│ step_1 -> start    ┆ 6054        ┆ 0       ┆ 6054        │
│ step_3 -> step_2   ┆ 3792        ┆ 0       ┆ 3792        │
│ confirm -> confirm ┆ 3621        ┆ 3621    ┆ 0           │
│ step_2 -> step_1   ┆ 3034        ┆ 0       ┆ 3034        │
│ …                  ┆ …           ┆ …       ┆ …           │
│ step_3 -> step_1   ┆ 978         ┆ 0       ┆ 978         │
│ confirm -> start   ┆ 838         ┆ 0       ┆ 838         │
│ confirm -> step_1  ┆ 126         ┆ 0       ┆ 126         │
│ confirm -> step_3  ┆ 105         ┆ 0       ┆ 105         │
│ confirm -> step_2  ┆ 1           ┆ 

In [24]:
# A visit is successful if every transition is a clean forward step (+1 in rank)
# Using shift: prev_step_rank + 1 == step_rank for every row in the visit

visit_success = (
    df_transitions
    .group_by('visit_id')
    .agg([
        pl.col('Variation').first().alias('Variation'),
        # All transitions must be clean forward steps
        (pl.col('step_rank') - pl.col('prev_step_rank')).alias('rank_diff'),
        pl.len().alias('n_transitions')
    ])
    .with_columns([
        # Successful = 5 steps means 4 transitions, all with rank_diff == 1
        (
            (pl.col('rank_diff').list.min() == 1) &
            (pl.col('rank_diff').list.max() == 1) &
            (pl.col('n_transitions') == 4)
        ).alias('successful')
    ])
)

total_visits = len(visit_success)
successful_visits = visit_success['successful'].sum()

print(f"\n=== Successful Visits ===")
print(f"Total unique visit_ids: {total_visits}")
print(f"Successful visits: {successful_visits}")
print(f"Overall success rate: {successful_visits / total_visits:.2%}")
print()
print(
    visit_success
    .group_by('Variation')
    .agg([
        pl.col('successful').sum().alias('successful'),
        pl.len().alias('total_visits')
    ])
    .with_columns(
        (pl.col('successful') / pl.col('total_visits')).alias('success_rate')
    )
    .sort('Variation')
)


=== Successful Visits ===
Total unique visit_ids: 43743
Successful visits: 15368
Overall success rate: 35.13%

shape: (2, 4)
┌───────────┬────────────┬──────────────┬──────────────┐
│ Variation ┆ successful ┆ total_visits ┆ success_rate │
│ ---       ┆ ---        ┆ ---          ┆ ---          │
│ str       ┆ u32        ┆ u32          ┆ f64          │
╞═══════════╪════════════╪══════════════╪══════════════╡
│ Control   ┆ 7037       ┆ 19106        ┆ 0.368314     │
│ Test      ┆ 8331       ┆ 24637        ┆ 0.33815      │
└───────────┴────────────┴──────────────┴──────────────┘


In [25]:
# First occurrence of each step per visit (regression rule: first entry wins)
df_first = (
    df_sorted
    .with_columns(
        pl.col('process_step').replace(STEP_ORDER).cast(pl.Int8).alias('step_rank')
    )
    .group_by(['visit_id', 'process_step'])
    .agg([
        pl.col('date_time').min().alias('step_start'),
        pl.col('step_rank').first().alias('step_rank'),
        pl.col('Variation').first().alias('Variation'),
    ])
    .sort(['visit_id', 'step_rank'])
)

# Shift to get next step's start time within each visit
df_first = df_first.with_columns([
    pl.col('step_start').shift(-1).over('visit_id').alias('next_step_start'),
    pl.col('process_step').shift(-1).over('visit_id').alias('next_step'),
])

avg_time = (
    df_first
    .filter(pl.col('next_step_start').is_not_null())
    .with_columns([
        (pl.col('next_step_start') - pl.col('step_start'))
        .dt.total_seconds()
        .alias('duration_seconds'),
        (pl.col('process_step') + ' -> ' + pl.col('next_step')).alias('transition')
    ])
    .filter(pl.col('duration_seconds') >= 0)
    .group_by('transition')
    .agg([
        pl.col('duration_seconds').mean().alias('avg_seconds'),
        pl.col('duration_seconds').median().alias('median_seconds'),
        pl.col('duration_seconds').std().alias('std_seconds'),
        pl.len().alias('n_visits')
    ])
    .with_columns([
        (pl.col('avg_seconds') / 60).round(2).alias('avg_minutes'),
        (pl.col('median_seconds') / 60).round(2).alias('median_minutes'),
    ])
    .sort('transition')
)

print("=== Average Time per Transition ===")
print(avg_time.select([
    'transition', 'n_visits',
    'avg_minutes', 'median_minutes', 'std_seconds'
]))

=== Average Time per Transition ===
shape: (9, 5)
┌───────────────────┬──────────┬─────────────┬────────────────┬─────────────┐
│ transition        ┆ n_visits ┆ avg_minutes ┆ median_minutes ┆ std_seconds │
│ ---               ┆ ---      ┆ ---         ┆ ---            ┆ ---         │
│ str               ┆ u32      ┆ f64         ┆ f64            ┆ f64         │
╞═══════════════════╪══════════╪═════════════╪════════════════╪═════════════╡
│ start -> step_1   ┆ 39133    ┆ 0.99        ┆ 0.28           ┆ 275.580583  │
│ start -> step_2   ┆ 14       ┆ 3.5         ┆ 1.56           ┆ 271.263264  │
│ start -> step_3   ┆ 44       ┆ 13.46       ┆ 11.31          ┆ 623.940006  │
│ step_1 -> confirm ┆ 5        ┆ 14.57       ┆ 9.02           ┆ 1236.298103 │
│ step_1 -> step_2  ┆ 33945    ┆ 1.07        ┆ 0.48           ┆ 154.724083  │
│ step_1 -> step_3  ┆ 41       ┆ 4.11        ┆ 2.9            ┆ 179.008502  │
│ step_2 -> confirm ┆ 35       ┆ 5.35        ┆ 4.5            ┆ 223.871887  │
│ step_2 -> st

In [26]:
# Step 1: count how many visit_ids each client_id has
client_visit_counts = (
    df_sorted
    .group_by('client_id')
    .agg(
        pl.col('visit_id').n_unique().alias('n_visits')
    )
)

# Step 2: from Block 3, get visit_ids that have zero errors
clean_visits = (
    df_transitions
    .group_by('visit_id')
    .agg([
        pl.col('has_error').any().alias('any_error'),
        pl.col('Variation').first().alias('Variation'),
        pl.len().alias('n_transitions')
    ])
    .filter(
        (pl.col('any_error') == False) &  # no errors
        (pl.col('n_transitions') == 4)     # exactly 4 transitions = all 5 steps
    )
)

# Step 3: keep only clients with a single visit_id
single_visit_clients = client_visit_counts.filter(pl.col('n_visits') == 1)

# Step 4: join — visit must be clean AND client must have only one visit
first_attempt_success = (
    clean_visits
    .join(
        df_sorted.select(['visit_id', 'client_id']).unique(),
        on='visit_id'
    )
    .join(
        single_visit_clients,
        on='client_id'
    )
)

# Summary
total_clients = df_sorted['client_id'].n_unique()
first_attempt_count = first_attempt_success['client_id'].n_unique()

print(f"Total unique clients: {total_clients}")
print(f"Completed start→confirm, first attempt, clean: {first_attempt_count}")
print(f"Rate: {first_attempt_count / total_clients:.2%}")
print()
print(
    first_attempt_success
    .group_by('Variation')
    .agg(pl.col('client_id').n_unique().alias('clients'))
    .with_columns(
        (pl.col('clients') / first_attempt_count).alias('share')
    )
    .sort('Variation')
)

Total unique clients: 40028
Completed start→confirm, first attempt, clean: 12201
Rate: 30.48%

shape: (2, 3)
┌───────────┬─────────┬──────────┐
│ Variation ┆ clients ┆ share    │
│ ---       ┆ ---     ┆ ---      │
│ str       ┆ u32     ┆ f64      │
╞═══════════╪═════════╪══════════╡
│ Control   ┆ 5656    ┆ 0.463569 │
│ Test      ┆ 6545    ┆ 0.536431 │
└───────────┴─────────┴──────────┘
